
# Agentic Multimodal Demo (Notebook)
Lightweight, agentic pipeline you can run step-by-step:
- **SeriesBuilder**: ordered people/things with dates (kings, CEOs, etc.) → SDXL portraits → poster
- **MapBuilder**: regions/points (Europe, US states, Canadian provinces, South America) → flags/icons → map

Repo layout assumption:
```
/notebooks/agentic_multimodal/agentic-multimodal.ipynb      # this notebook (root)
/src/demos/agentic_multimodal #supporting modules
/results/agentic_multimodal    # output data
```


#### Load all the relevant code into a registry

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent.parent.resolve()
SRC = ROOT / "src" / "demos"
sys.path.insert(0, str(SRC))
from agentic_multimodal.core.config import CACHE, RESULTS, SRC
from agentic_multimodal.schemas.artifacts import PosterSpec, PosterItem, ImageAsset
from agentic_multimodal.services.registry import make_registry
from agentic_multimodal.skills.image_gen import generate_person_images
from agentic_multimodal.skills.gen_poster_renderer import compose_poster_spec
from agentic_multimodal.skills.adapters import (
    countries_to_mapspec,
    image_from_flag_url,
    image_from_pick_url,
    label_country,
    label_country_plus_pick,
    name_year_pairs,
    render_portraits_for_picks,
)

from support import tree_markdown
import os, re

MANIFEST = ROOT / "results" / "agentic_multimodal" / "manifest.jsonl"
reg = make_registry(ROOT)

BUILD_TEASER = False
BUILD_POTUS = False
BUILD_EUROPE = False
BUILD_EUROPE_PORTRAITS = False
BUILD_NOBEL_PHYSICS = False
BUILD_MONARCHS_ENG_GB_UK = False


/opt/anaconda3/envs/ai_backup_py312/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [2]:
# Initialize registry adapters
reg.adapters.countries_to_mapspec = countries_to_mapspec
reg.adapters.image_from_flag = image_from_flag_url
reg.adapters.image_from_pick = image_from_pick_url
reg.adapters.label_country = label_country
reg.adapters.label_country_plus_pick = label_country_plus_pick
reg.adapters.render_portraits_for_picks = render_portraits_for_picks


In [3]:

for dir in [CACHE, RESULTS]:
    dir.mkdir(parents=True, exist_ok=True)
    if str(dir) not in sys.path:
        sys.path.insert(0, str(dir))

# Show code tree from source:
display(tree_markdown(SRC))

```agentic_multimodal/
├── assets
│   ├── ne
│   │   ├── ne_110m_admin_0_countries.geojson
│   │   └── ne_50m_admin_0_countries.geojson
│   └── prompts
│       └── overrides.yaml
├── cache
├── core
│   ├── __init__.py
│   └── config.py
├── graphs
│   ├── __init__.py
│   ├── factory.py
│   ├── geo_flow.py
│   ├── group_poster_flow.py
│   └── person_flow.py
├── schemas
│   ├── __init__.py
│   ├── artifacts.py
│   ├── entities.py
│   ├── messages.py
│   └── requests.py
├── services
│   ├── cache.py
│   ├── io_web_fetcher.py
│   ├── llm_factory.py
│   ├── registry.py
│   └── settings.py
├── skills
│   ├── adapters
│   │   ├── __init__.py
│   │   ├── countries_to_map.py
│   │   ├── items_to_poster.py
│   │   ├── people_to_poster.py
│   │   └── picks_to_portraits.py
│   ├── data
│   │   ├── __init__.py
│   │   ├── natural_earth.py
│   │   ├── wd_utils.py
│   │   ├── wiki_pageviews.py
│   │   ├── wikidata_geo.py
│   │   ├── wikidata_search_label.py
│   │   ├── wikidata_series.py
│   │   ├── wikidata_sparql.py
│   │   ├── wikidata_taxa.py
│   │   └── wikipedia_category_members.py
│   ├── geo
│   │   ├── __init__.py
│   │   ├── aliases.py
│   │   ├── europe_flags.py
│   │   ├── flight_path.py
│   │   ├── region_flags.py
│   │   └── subdivisions.py
│   ├── selectors
│   │   ├── __init__.py
│   │   └── famous_by_country.py
│   ├── series
│   │   ├── __init__.py
│   │   ├── aliases.py
│   │   ├── award.py
│   │   ├── candidates.py
│   │   ├── group_items.py
│   │   ├── per_country.py
│   │   ├── positions.py
│   │   └── rankers.py
│   ├── __init__.py
│   ├── gen_map_renderer.py
│   ├── gen_poster_renderer.py
│   ├── image_gen.py
│   ├── image_prompts.py
│   └── map_regions.py
├── __init__.py
└── notebook_utils.py
```

## Poster compositor (grid + captions)


In [ ]:
import hashlib
import re
from agentic_multimodal.schemas.artifacts import PosterSpec, PosterItem
from agentic_multimodal.skills.adapters.people_to_poster import _asset_from_path
from agentic_multimodal.skills.image_gen import generate_prompt_images
import numpy as np

def _clean_title(t: str) -> str:
    # "Robin (bird)" -> "Robin"
    return re.sub(r"\s*\(.*\)\s*$", "", (t or "")).strip()

style_by_group = {
  "bird": "wildlife photography, wings and feathers detailed",
  "reptile": "wildlife photography, scales detailed",
  "mammal": "wildlife photography, fur detailed",
}

def wdqs_category_qids(category_title: str, limit: int = 300):
    q = f"""
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX mwapi: <https://www.mediawiki.org/ontology#API/>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT DISTINCT ?item ?wikiTitle WHERE {{
  SERVICE wikibase:mwapi {{
    bd:serviceParam wikibase:endpoint "en.wikipedia.org" ;
                    wikibase:api "Generator" ;
                    wikibase:limit "once" ;
                    mwapi:generator "categorymembers" ;
                    mwapi:gcmtitle "{category_title}" ;
                    mwapi:gcmnamespace "0" ;
                    mwapi:gcmlimit "{limit}" .
    ?item      wikibase:apiOutputItem mwapi:item .
    ?wikiTitle wikibase:apiOutput mwapi:title .
  }}

  FILTER(BOUND(?item))
}}
LIMIT {limit}
"""
    return reg.sparql.run(q)

def wdqs_enrich_names(qids: list[str]):
    values = " ".join(f"wd:{qid}" for qid in qids)

    q = f"""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT ?item ?itemLabel ?commonName ?taxonName WHERE {{
  VALUES ?item {{ {values} }}

  OPTIONAL {{ ?item wdt:P1843 ?commonName . FILTER(LANG(?commonName) = "en") }}
  OPTIONAL {{ ?item wdt:P225  ?taxonName }}

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
"""
    rows = reg.sparql.run(q)

    out = {}
    for r in rows:
        qid = r["item"]["value"].rsplit("/", 1)[-1]
        out[qid] = {
            "label": r.get("itemLabel", {}).get("value"),
            "common": r.get("commonName", {}).get("value"),
            "taxon": r.get("taxonName", {}).get("value"),
        }
    return out


def rows_to_items(rows, group: str):
    out = []
    for r in rows:
        qid = r["item"]["value"].rsplit("/", 1)[-1]

        wiki = _clean_title(r.get("wikiTitle", {}).get("value", "") or "")
        common = (r.get("commonName") or {}).get("value")
        sci = (r.get("taxonName") or {}).get("value") or r.get("itemLabel", {}).get("value")

        display = common or wiki or sci
        out.append({"qid": qid, "display": display, "scientific": sci, "group": group})
    return out


def take_by_group(pool, targets: dict, seed: str):
    # deterministic per-group sampling so the mix stays stable
    out = []
    for grp, n in targets.items():
        sub = [x for x in pool if x["group"] == grp]
        out.extend(stable_take(sub, n, seed=f"{seed}:{grp}"))
    return out

def animal_prompt(display: str, scientific: str | None, group: str) -> str:
    tag = f"{display} ({scientific})" if scientific and scientific.lower() != display.lower() else display
    group_hint = {
        "bird": "feathers, beak, wings",
        "reptile": "scales, reptile skin texture",
        "mammal": "fur, whiskers",
    }.get(group, "")
    return (
        f"Wildlife photograph of {tag}. "
        f"EXACTLY ONE ANIMAL. single subject only. one instance only. "
        f"Centered, full body visible, sharp focus, "
        f"{group_hint}."
        f"No other animals. No duplicate subjects. No repetition."
    )


# ----------------------------
# 1) Fetch pools (fast)
# ----------------------------
def clean_title(t: str) -> str:
    return re.sub(r"\s*\(.*\)\s*$", "", (t or "")).strip()

def stable_take(items, n: int, seed: str):
    def k(it):
        return hashlib.md5((it["qid"] + seed).encode("utf-8")).hexdigest()
    return sorted(items, key=k)[:n]

cats = {
    "mammal":  "Category:Mammals of the United States",
    "bird":    "Category:Birds of the United States",
    "reptile": "Category:Reptiles of the United States",
}

# Step A: fast pull
pool = []
for group, cat in cats.items():
    rows = wdqs_category_qids(cat, limit=350)
    for r in rows:
        qid = r["item"]["value"].rsplit("/", 1)[-1]
        title = clean_title(r.get("wikiTitle", {}).get("value", ""))
        pool.append({"qid": qid, "wiki": title, "group": group})

# dedupe by qid
dedup = {}
for it in pool:
    dedup.setdefault(it["qid"], it)
pool = list(dedup.values())

# ----------------------------
# 2) Pick 100 with a stable mix
# ----------------------------
N = 20
seed = "animals_mix_v1"

targets = {"mammal": N//3 + N%3, "bird": N//3, "reptile": N//3}
picked = []
for g, n in targets.items():
    picked.extend(stable_take([x for x in pool if x["group"] == g], n, seed=f"animals_mix_v1:{g}"))
picked = picked[:N]

# Step B: enrich only the chosen
enriched = wdqs_enrich_names([x["qid"] for x in picked])

# build final items
items = []
for it in picked:
    meta = enriched.get(it["qid"], {})
    display = meta.get("common") or it["wiki"] or meta.get("taxon") or meta.get("label") or it["qid"]
    sci = meta.get("taxon") or meta.get("label")
    items.append({**it, "display": display, "scientific": sci})

# Safety: if a group returns short, top up from remaining
if len(items) < N:
    used = {x["qid"] for x in items}
    rest = [x for x in stable_take(pool, 1000, seed=seed+":topup") if x["qid"] not in used]
    items.extend(rest[: (N - len(items))])

print("Counts:", {g: sum(1 for x in items if x["group"] == g) for g in targets})
print("Sample:", [x["display"] for x in items[:8]])

# ----------------------------
# 3) Generate images
# ----------------------------
# prompts
negative = (
    "text, letters, watermark, logo, caption, typography, "
    "frame, border, matting, photo frame, picture frame, "
    "clothing, shirt, suit, tie, collar, dress, buttons, "
    "human, person, hands, arms, mannequin"
)

picks = [
    {
        "qid": it["qid"],
        "name": it["display"],
        "prompt": animal_prompt(it["display"], it["scientific"], it["group"]),
    }
    for it in items
]

paths = generate_prompt_images(
    picks,
    outdir="artifacts/animals_portraits",
    size=(512, 512),
    steps=28,
    cfg=6.0,
    seed=1337,
    skip_existing=True,
    negative_prompt=negative,
)

# ----------------------------
# 4) Build poster spec + render
# ----------------------------
poster_items = [
    PosterItem(
        image=_asset_from_path(p),
        #label=f"{it['display']}\n{it['scientific']}" if it["scientific"] and it["scientific"].lower() != it["display"].lower() else it["display"]
        label=f"{it['display']}"
    )
    for it, p in zip(items, paths)
]

spec = PosterSpec(
    title=f"{N} Animals (Mammals • Birds • Reptiles)",
    grid_cols=int(np.sqrt(N)),
    items=poster_items,
)

reg.render.poster(
    spec,
    tile_size=(512, 512),
    outpath=f"artifacts/poster_animals_{N}_mixed.webp",
)


Counts: {'mammal': 8, 'bird': 6, 'reptile': 6}
Sample: ['North American cougar', 'Pacific Water Shrew', 'White-backed Hog-nosed Skunk', 'Southern broad-footed mole', 'Dall’s White-flanked Porpoise', 'Cockrum’s Grey Shrew', 'Semirostrum ceruttii', 'Alaska moose']


Keyword arguments {'add_watermark': False} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

'artifacts/poster_animals_100_mixed.webp'

In [5]:
if BUILD_POTUS:
    people = reg.series.run("potus")
    pairs = name_year_pairs(people, mode="per_term")
    paths = generate_person_images(pairs, outdir="artifacts/potus_terms_portraits")
    spec  = reg.adapters.people_per_term(people, title="U.S. Presidents — Terms", image_paths=paths, cols=6)
    reg.render.poster(spec, outpath="artifacts/poster_presidents_terms.webp")

In [6]:
from agentic_multimodal.skills.series.group_items import get_group_items
from agentic_multimodal.skills.adapters.items_to_poster import items_to_posterspec
from agentic_multimodal.skills.image_gen import generate_prompt_images
from agentic_multimodal.skills.image_prompts import subject_portrait

items = get_group_items(reg, item_type="animals", count=1, seed="seed42")

picks = [
    {
        "name": it.name,
        "prompt": subject_portrait(subject_type="animals", name=it.name, style="photo", no_text=True),
    }
    for it in items
]

paths = generate_prompt_images(picks, outdir="artifacts/animals_portraits", size=(768, 768))
spec  = items_to_posterspec(items, title="25 Animals", image_paths=paths, cols=2)

reg.render.poster(spec, outpath="artifacts/poster_100_animals.webp", tile_size=(768, 768))


Keyword arguments {'add_watermark': False} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

'artifacts/poster_100_animals.webp'

In [7]:
if BUILD_NOBEL_PHYSICS:
    laureates = reg.series.run("nobel_physics")
    pairs = name_year_pairs(laureates, mode="per_person_auto")
    paths = generate_person_images(pairs, outdir="artifacts/nobel_physics_portraits")
    spec  = reg.adapters.people_per_person(laureates, title="Nobel Prize in Physics — Laureates", image_paths=paths, cols=12)
    reg.render.poster(spec, outpath="artifacts/poster_nobel_physics.webp")


In [8]:
if BUILD_MONARCHS_ENG_GB_UK:
    people = reg.series.run("monarchs_eng_gb_uk")
    pairs = name_year_pairs(people, mode="per_term")
    paths = generate_person_images(pairs, outdir="artifacts/monarchs_terms_portraits")
    spec  = reg.adapters.people_per_term(people, title="UK Monarchs — Terms", image_paths=paths, cols=8)
    reg.render.poster(spec, outpath="artifacts/poster_monarchs_terms.webp")

In [9]:
# ---- Teaser poster with merged term spans ----
if BUILD_TEASER:

    teaser_names = [
        "George Washington",
        "Abraham Lincoln",
        "John F. Kennedy",
        "Ronald Reagan",
        "Barack Obama",
        "Donald Trump",
    ]

    # 1) Look up term years from your series data
    people = reg.series.run("potus")

    def _y(s):
        if not s: return None
        s = s.lstrip("+")
        return s[:4] if len(s) >= 4 and s[:4].isdigit() else None

    def term_spans(person):
        # Build "YYYY–YYYY" or "YYYY–PT"; sort by start year
        spans = []
        for t in getattr(person, "terms", []):
            ys = _y(getattr(t, "start", None))
            ye = _y(getattr(t, "end", None))
            if ys or ye:
                spans.append((ys, ye))
        # sort by numeric start, then end
        spans.sort(key=lambda ab: (int(ab[0]) if (ab[0] and ab[0].isdigit()) else 9999,
                                int(ab[1]) if (ab[1] and ab[1].isdigit()) else 9999))
        # format with en dash
        out = []
        for ys, ye in spans:
            if ys and ye:
                out.append(f"{ys}–{ye}")
            elif ys and not ye:
                out.append(f"{ys}–PT")
            elif not ys and ye:
                out.append(f"–{ye}")
        return ", ".join(out)

    by_name = {p.name: p for p in people}

    # 2) Map names → existing portrait paths (first match per name)
    IMG_DIR = "artifacts/potus_terms_portraits"   # adjust if needed

    def _canon(s): return re.sub(r"\W+", "", s).lower()
    paths_by_name = {}
    for fn in sorted(os.listdir(IMG_DIR)):
        if not fn.lower().endswith((".png",".jpg",".jpeg",".webp")):
            continue
        name_part = re.sub(r"^\d+_", "", os.path.splitext(fn)[0])
        paths_by_name.setdefault(_canon(name_part), os.path.join(IMG_DIR, fn))

    def _portrait_for(name):
        p = paths_by_name.get(_canon(name))
        if not p:
            raise FileNotFoundError(f"No portrait found for '{name}' in {IMG_DIR}")
        return p

    # 3) Build items with NAME + merged term spans
    items = []
    for nm in teaser_names:
        label_lines = [nm]
        if nm in by_name:
            spans = term_spans(by_name[nm])
            if spans:
                label_lines.append(spans)
        label = "\n".join(label_lines)

        items.append(
            PosterItem(
                image=ImageAsset(id=_canon(nm), path=_portrait_for(nm), width=512, height=768),
                label=label
            )
        )

    # 4) 3 columns × 2 rows; export sized for LinkedIn (landscape-ish)
    spec_teaser = PosterSpec(
        title="U.S. Presidents — Agentic Poster (Teaser)",
        grid_cols=3,
        items=items
    )

    out_teaser = "artifacts/linkedin/potus_teaser_3x2_terms_1620x1080.jpg"
    os.makedirs(os.path.dirname(out_teaser), exist_ok=True)

    # Slightly larger captions help on mobile; tune if your renderer exposes these knobs
    compose_poster_spec(
        spec_teaser,
        outpath=out_teaser,
        out_format="JPEG",
        out_quality=88,
        max_long_side=1620,          # ~1620×1080 export
        # If your renderer supports it, these help readability:
        # caption_scale=2.2,
        # line_gap_px=8,
        # stroke_width_px=3,
    )

    print("Teaser saved:", out_teaser)



## MapBuilder: regions + capitals + flags

In [10]:
def image_from_pick_path(country, pick):
    path = portrait_paths.get(country.qid)
    return {"path": path} if path else None


if BUILD_EUROPE:
    countries = reg.geo.run("europe_countries_flags")

    spec = countries_to_mapspec(
        reg.geo.run("europe_countries_flags"),
        title="Europe — Flags at Capitals",
        region_key="europe_flags",
    )

    reg.render.map(
        spec,
        outdir="artifacts/maps",
        size=(2200, 1320),
        marker_px=42,              
        show_labels=True,        
        show_country_names=True,
        show_capital_names=False,
        min_flag_separation_px=48,
        min_label_separation_px=96,
        max_labels=None,
    )
    print("Saved:", spec.path)


In [11]:
if BUILD_EUROPE_PORTRAITS:
    PORTRAIT_TYPE = 'sports' # sports, musicians
    person_type = PORTRAIT_TYPE if PORTRAIT_TYPE == 'musicians' else 'sportsperson'
    N_countries = 15
    countries = reg.geo.run("europe_countries_flags", min_pop=2000000)
    topN = sorted(countries, key=lambda c: c.population or 0, reverse=True)[:N_countries]

    picks = {}
    for c in topN:
        ppl = reg.geo.run(
            "famous_by_country",
            country_qid=c.qid,
            category=PORTRAIT_TYPE,
            limit_candidates=60,
            days=90,
            score="blended",   # ← new
        )
        best = ppl[0] if ppl else None
        if best:
            picks[c.qid] = {"qid": best.qid, "label": best.name, "image_url": best.image_url}

    portrait_paths = render_portraits_for_picks(picks, outdir=f"artifacts/europe_top{N_countries}_{person_type}_portraits")

    # attach local paths back onto the picks
    for qid, p in picks.items():
        lp = portrait_paths.get(qid)
        if lp:
            if isinstance(p, dict):
                p["local_image_path"] = lp
            else:
                setattr(p, "local_image_path", lp)  # if it’s a model you control

    spec = countries_to_mapspec(
        topN,
        title=f"Europe — most popular {person_type} (top-{N_countries} countries)",
        picks=picks,
        label_fn=label_country_plus_pick,
        image_fn = lambda c, p: {"path": p.get("local_image_path")} \
        if isinstance(p, dict) and p.get("local_image_path") else None
    )


    img = reg.render.map(
        spec,
        show_country_names=True,
        show_labels=True,
        show_pick_images=True,          # portrait bubbles on the capitals
        pick_image_size_px=125,         
        show_flag_markers=False,        # portraits > flags here
    )
    display(img)



In [12]:
if 0:
    # 1) Get U.S. states with flags + capital coords (you already registered this alias)
    states = reg.geo.run("us_states_flags")  # SubdivisionsByCountryProvider under the hood

    # 2) Keep the West. Adjust the list if you want to be stricter/looser.
    WEST_ABBR = {
        "WA","OR","CA","NV","ID","MT","WY","UT","CO","AZ","NM","AK","HI"
    }
    west_states = [s for s in states if getattr(s, "abbrev", None) in WEST_ABBR]

    # 3) Build the map spec: labels optional; flags as the marker image
    from agentic_multimodal.skills.adapters import countries_to_mapspec, image_from_flag_url, label_country

    spec = countries_to_mapspec(
        west_states,
        title="Western U.S. — State Flags",
        picks=None,                              # no per-state “pick” objects
        label_fn=lambda s, _: s.name,            # or 'label_country' if you prefer
        image_fn=lambda s, _: {"url": s.flag_svg_url},  # <- use the flag as marker image
        region="Western U.S."
    )

    # 4) Render. Portraits off, flags on. Keep dots tiny as fallback.
    img_path = reg.render.map(
        spec,
        show_country_names=False,        # hide text if you want a clean flag-only look
        show_labels=False,               # turn on if you want state names under flags
        show_pick_images=False,          # we’re not using portraits here
        show_flag_markers=True,          # <- show the flags
        flag_marker_size_px=42,          # tweak 36–56 based on your canvas
        marker_px=2,                     # tiny dot fallback
    )
    display(img_path)

In [13]:
#artifact = reg.graphs.geo.invoke({"question": "Show me the flight path from Seattle, WA to London, UK"})
#artifact = reg.graphs.geo.invoke({"question": "Show me the flight path from Miami, FL to Oslo, Norway"})
#artifact = reg.graphs.geo.invoke({"question": "Show me the flight path from Paris, TX to Honolulu, HI"})
artifact = reg.graphs.geo.invoke({"question": "Show me the flight path from Rome, GA to Rome, Italy"})
print("Saved:", artifact.get("artifact_path"))


Saved: artifacts/maps/Flight Path_ Rome_ GA _ Rome_ Italy __5_038 mi_.png
